In [30]:
import pandas as pd
import numpy as np

In [31]:
df=pd.read_csv(r"C:\Users\Rohan Landge\Downloads\NER dataset.csv", encoding="latin-1")
df

,Sentence #,Word,POS,Tag
0,Sentence: 1,Thousands,NNS,O
1,NaN,of,IN,O
2,NaN,demonstrators,NNS,O
3,NaN,have,VBP,O
4,NaN,marched,VBN,O
...,...,...,...,...
1048570,NaN,they,PRP,O
1048571,NaN,responded,VBD,O
1048572,NaN,to,TO,O
1048573,NaN,the,DT,O


In [32]:
df.isnull().sum()

Sentence #    1000616
Word               10
POS                 0
Tag                 0
dtype: int64

In [33]:
df=df[["Sentence #","Word","POS"]]
df=df.fillna(method='ffill')

C:\Users\Rohan Landge\AppData\Local\Temp\ipykernel_26524\1816641520.py:2: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df=df.fillna(method='ffill')


In [34]:
sentences=df.groupby('Sentence #')['Word'].apply(list).values
tags=df.groupby('Sentence #')['POS'].apply(list).values

In [35]:
df.isnull().sum()

Sentence #    0
Word          0
POS           0
dtype: int64

In [36]:
from tensorflow.keras.preprocessing.text import Tokenizer
sen_tokenizer=Tokenizer()
tag_tokenizer=Tokenizer()

sen_tokenizer.fit_on_texts(sentences)
tag_tokenizer.fit_on_texts(tags)

x=sen_tokenizer.texts_to_sequences(sentences)
y=tag_tokenizer.texts_to_sequences(tags)

In [37]:
from tensorflow.keras.preprocessing.sequence import pad_sequences
max_len=128
x=pad_sequences(x,maxlen=max_len,padding='post')
y=pad_sequences(y,maxlen=max_len, padding='post')

In [38]:
from sklearn.model_selection import train_test_split
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2)

In [39]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM,Embedding,Dense,TimeDistributed

In [40]:
model=Sequential()

model.add(Embedding(
    input_dim=len(sen_tokenizer.word_index)+1,
    output_dim=64
))

model.add(LSTM(64,return_sequences=True))

model.add(TimeDistributed(Dense(
    len(tag_tokenizer.word_index)+1,
    activation='softmax'
)))

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [41]:
model.fit(x_train,y_train,batch_size=32,epochs=2)

Epoch 1/2
1199/1199 ━━━━━━━━━━━━━━━━━━━━ 254s 201ms/step - accuracy: 0.9542 - loss: 0.2038
Epoch 2/2
1199/1199 ━━━━━━━━━━━━━━━━━━━━ 254s 194ms/step - accuracy: 0.9927 - loss: 0.0260


In [42]:
loss,acc=model.evaluate(x_test,y_test)
print('Accuracy :',acc)

300/300 ━━━━━━━━━━━━━━━━━━━━ 20s 57ms/step - accuracy: 0.9925 - loss: 0.0240
Accuracy : 0.9924766421318054


In [43]:
index_to_tag = {v:k for k,v in tag_tokenizer.word_index.items()}

test = pad_sequences(sen_tokenizer.texts_to_sequences([sentences[0]]), maxlen=128, padding='post')

pred = np.argmax(model.predict(test), axis=-1)[0]

print("\nWord -> Predicted POS Tag:\n")

for w, p in zip(sentences[0], pred):
    print(w, '->', index_to_tag.get(p, "O"))

1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step

Word -> Predicted POS Tag:

Thousands -> nns
of -> in
demonstrators -> nns
have -> vbp
marched -> vbn
through -> in
London -> nnp
to -> to
protest -> vb
the -> dt
war -> nn
in -> in
Iraq -> nnp
and -> cc
demand -> nn
the -> dt
withdrawal -> nn
of -> in
British -> jj
troops -> nns
from -> in
that -> in
country -> nn
. -> .
